mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path to the AgentIF data

In [ ]:
AGENTIF_FILE_PATH = "/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_with_attention_scores_probe_Answer: _model_Qwen2.5-7B_DOWNSAMPLED.pkl"

imports

In [ ]:
import warnings

# remove warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import json
from pathlib import Path
from copy import deepcopy
from tqdm.notebook import tqdm
import re


aux model to get attention scores

In [ ]:
def _sent_spans(text: str):
    # returns [(char_start, char_end, span_text), ...]
    spans = []
    for m in re.finditer(r".+?(?:[.!?](?:\s+|$)|\n{2,}|$)", text, flags=re.S):
        s = m.group(0)
        if s.strip():
            spans.append((m.start(), m.end(), s))
    return spans

def _approx_tokens(s: str) -> int:
    return max(1, len(s) // 4)  # cheap heuristic; good enough for budgeting

@torch.no_grad()
def get_attention_scores(context, objective, *, top_layers=8, max_seq_len=2048, ctx_stride=256):
    context = context.strip()
    objective = objective.strip()
    if not context:
        return []

    # 1) tokenize objective once
    enc_obj = _aux_tok(objective, add_special_tokens=False, return_tensors="pt")
    obj_ids = enc_obj["input_ids"][0]

    # 2) tokenize context once (with offsets)
    enc_ctx = _aux_tok(context, add_special_tokens=False, return_tensors="pt",
                       return_offsets_mapping=True)
    ctx_ids = enc_ctx["input_ids"][0]
    ctx_offsets = enc_ctx["offset_mapping"][0].tolist()

    # 3) sep + chunk size
    sep_ids = _aux_tok("\n\n", add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    sep_len = sep_ids.numel()
    max_ctx_chunk = max(32, max_seq_len - (obj_ids.numel() + sep_len + 2))

    device = _aux_model.device

    # 4) CPU accumulator
    imp_cpu = torch.zeros((ctx_ids.numel(),), dtype=torch.float32)

    start = 0
    while start < ctx_ids.numel():
        end = min(ctx_ids.numel(), start + max_ctx_chunk)
        chunk_ids = ctx_ids[start:end]

        # 5a) build input: chunk + sep + objective
        input_ids = torch.cat([chunk_ids, sep_ids, obj_ids], dim=0).unsqueeze(0).to(device)
        attn_mask = torch.ones_like(input_ids)

        ctx_len_with_sep = chunk_ids.numel() + sep_len
        seq_len = input_ids.shape[1]
        q_pos = list(range(ctx_len_with_sep, seq_len))  # objective token positions

        # 5b) forward (big memory happens here, but seq_len is capped)
        out = _aux_model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True,
            use_cache=False,
            return_dict=False,
        )
        # depending on model, attentions are out[-1] or out[2]; safest:
        attns = out[-1]

        # 5c) compute importance for this chunk (on GPU, small)
        L = len(attns)
        l0 = max(0, L - top_layers)
        imp_chunk = torch.zeros((ctx_len_with_sep,), device=device)

        for l in range(l0, L):
            a = attns[l][0]  # (heads, seq, seq)
            imp_chunk += a[:, q_pos, :ctx_len_with_sep].sum(dim=(0, 1))

        # 5d) drop sep tokens, move to CPU, add into global accumulator
        imp_ctx_only_cpu = imp_chunk[:chunk_ids.numel()].float().cpu()
        imp_cpu[start:end] += imp_ctx_only_cpu

        # 5e) cleanup to avoid accumulation across chunks
        del out, attns, imp_chunk, imp_ctx_only_cpu, input_ids, attn_mask, a
        torch.cuda.empty_cache()
        gc.collect()

        if end == ctx_ids.numel():
            break
        start = end - ctx_stride  # overlap

    # 6) normalize once
    imp_cpu = imp_cpu / (imp_cpu.sum() + 1e-9)

    # 7) token scores -> sentence/span scores
    spans = _sent_spans(context)  # returns (char_s, char_e, text)
    # map token scores to spans via offsets overlap
    scored = []
    for i, (sp_s, sp_e, sp_txt) in enumerate(spans):
        s = 0.0
        for tok_i, (ts, te) in enumerate(ctx_offsets):
            if te <= sp_s:
                continue
            if ts >= sp_e:
                break
            if te > ts and ts < sp_e and te > sp_s:
                s += float(imp_cpu[tok_i].item())
        scored.append((i, s, sp_txt))

    return scored

def compress_with_attention_scores_percentage(
    scored_spans: list,
    *,
    proportion_keep: float = 0.50,
    keep_first_span: bool = True, # keeps Goal/system-ish preface
    ) -> str:

    """
    Given attention scores for each span, keeps the top scoring proportion_keep percentage of sentences and cuts the rest
    """
    n_total = len(scored_spans)

    # Select spans under budget
    keep = set()
    if keep_first_span and n_total > 0:
        keep.add(0)

    #figure out how many sentences to keep
    n_keep = int(round(n_total * proportion_keep))
    if proportion_keep > 0:
        n_keep = max(1, n_keep)
    else:
        n_keep = 0

    #reserve space for first span if needed
    if keep_first_span and 0 in keep and n_keep > 0:
        remaining = max(0, n_keep - 1)
    else:
        remaining = n_keep

    #figure out which to keep
    for i, score, txt in sorted(scored_spans, key=lambda x: x[1], reverse=True):
        if i in keep:
            continue
        if remaining <= 0:
            break
        keep.add(i)
        remaining -= 1

    return "\n".join(scored_spans[i][2].strip() for i in sorted(keep)) #in original order

def compress_text_percentage(
    context: str,
    objective: str,
    *,
    proportion_keep: float = 0.50,
    top_layers: int = 8,          # use last N layers for "more semantic" attention
    keep_first_span: bool = True, # keeps Goal/system-ish preface
):
  scored = get_attention_scores(context, objective, top_layers=top_layers)
  compressed = compress_with_attention_scores_percentage(scored,proportion_keep=proportion_keep,keep_first_span=keep_first_span)
  return compressed

load the AgentIF data

In [ ]:
p = Path(AGENTIF_FILE_PATH)

def load_agentif_rows(path: Path):
    if path.suffix == ".pkl":
        obj = pd.read_pickle(path)  # could be DataFrame or list[dict]
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict("records")
        return obj
    elif path.suffix == ".json":
        return json.loads(path.read_text())
    elif path.suffix in [".jsonl", ".jl"]:
        return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

rows = load_agentif_rows(p)
print("Loaded rows:", len(rows))
print("Example rows:", rows[1])
print("Example keys:", rows[1].keys())
print("Example values:", rows[1].values())

save the initial prompts

In [ ]:
init_proportion = 0.50

In [ ]:
BATCH_SIZE = 32

batch = []

all_prompts = []
for r in tqdm(rows): #runs agentic system on all of the outputs
    rr = deepcopy(r)
    scored_spans = rr["scored_spans"]
    full_input = compress_with_attention_scores_percentage(scored_spans, proportion_keep = init_proportion)
    all_prompts.append({"prompt":full_input})

In [ ]:
def convert_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

In [ ]:
EXPERIMENT_NUM = "4"
AGENT_OUTPUT_PATH_JSON = Path("/content/drive/MyDrive/TokenReductionProject/Results/Prompts/experiment_" + EXPERIMENT_NUM +".json")
AGENT_OUTPUT_PATH_JSON.write_text(json.dumps(all_prompts, default=convert_numpy, ensure_ascii=False, indent=2))
print("Wrote", AGENT_OUTPUT_PATH_JSON, "with rows:", len(all_prompts))

In [ ]:
AGENT_OUTPUT_PATH_PKL = Path("/content/drive/MyDrive/TokenReductionProject/Results/Prompts/experiment_" + EXPERIMENT_NUM +".pkl")

pkl_df = pd.DataFrame(all_prompts)
pkl_df.to_pickle(AGENT_OUTPUT_PATH_PKL)

print("Wrote", AGENT_OUTPUT_PATH_PKL, "with rows:", len(all_prompts))